<a href="https://colab.research.google.com/github/dubeyvaibhav791-cloud/Diabetic_Readmission_Prediction/blob/main/FRAUD_DETECTION.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# Import libraries
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report
from xgboost import XGBClassifier

# 1. Load the dataset
data = pd.read_csv("/content/train_transaction.csv")

print("Dataset loaded successfully")
print(data.head())

# 2. Select features
features = [
    "TransactionAmt",
    "card1",
    "card2",
    "card3",
    "card5",
    "addr1",
    "C1",
    "C2",
    "C5",
    "C13"
]

X = data[features]
y = data["isFraud"]

# 3. Check missing values
print("\nMissing Values Before Filling:")
print(X.isnull().sum())

# 4. Fill missing values with median
X = X.fillna(X.median())

print("\nMissing Values After Filling:")
print(X.isnull().sum())

# 5. Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# 6. Check class distribution
normal = (y_train == 0).sum()
fraud = (y_train == 1).sum()

weight = normal / fraud

print("\nNormal Transactions:", normal)
print("Fraud Transactions:", fraud)
print("Scale Pos Weight:", round(weight, 2))

# 7. Create XGBoost model
model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    scale_pos_weight=weight,
    eval_metric="auc",
    random_state=42
)

# 8. Train model
model.fit(X_train, y_train)

# 9. Predict fraud probability
probability = model.predict_proba(X_test)[:, 1]

# 10. Calculate ROC-AUC
auc = roc_auc_score(y_test, probability)

print("\nROC-AUC Score:", round(auc, 4))

# 11. Final predictions
prediction = (probability >= 0.5).astype(int)

# 12. Classification Report
print("\nClassification Report:")
print(classification_report(y_test, prediction))

# 13. Feature Importance
importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': model.feature_importances_
}).sort_values(by='Importance', ascending=False)

print("\nFeature Importance:")
print(importance)

Dataset loaded successfully
   TransactionID  isFraud  TransactionDT  TransactionAmt ProductCD  card1  \
0        2987000        0          86400            68.5         W  13926   
1        2987001        0          86401            29.0         W   2755   
2        2987002        0          86469            59.0         W   4663   
3        2987003        0          86499            50.0         W  18132   
4        2987004        0          86506            50.0         H   4497   

   card2  card3       card4  card5  ... V330  V331  V332  V333  V334 V335  \
0    NaN  150.0    discover  142.0  ...  NaN   NaN   NaN   NaN   NaN  NaN   
1  404.0  150.0  mastercard  102.0  ...  NaN   NaN   NaN   NaN   NaN  NaN   
2  490.0  150.0        visa  166.0  ...  NaN   NaN   NaN   NaN   NaN  NaN   
3  567.0  150.0  mastercard  117.0  ...  NaN   NaN   NaN   NaN   NaN  NaN   
4  514.0  150.0  mastercard  102.0  ...  0.0   0.0   0.0   0.0   0.0  0.0   

  V336  V337  V338  V339  
0  NaN   NaN   NaN 